# Convolutional Kernel Network
Notebook pour entraîner et évaluer un CKN sur CIFAR-10 (split train/dev de Xtr et Ytr).

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import math
import scipy
import scipy.signal
import scipy.optimize
from timeit import default_timer as timer
import pickle

## 2. Utilitaires

In [2]:
EPS = 1e-6

def normalize_np(x, p=2, axis=-1):
    norm = np.linalg.norm(x, ord=p, axis=axis, keepdims=True)
    x /= np.maximum(norm, EPS)
    return x

def gaussian_filter(size, sigma=None):
    if size == 1:
        return np.ones(1)
    if sigma is None:
        sigma = (size - 1.) / (2. * math.sqrt(2))
    m = (size - 1) / 2.
    filt = np.arange(-m, m + 1)
    filt = np.exp(-filt ** 2 / (2. * sigma ** 2))
    return filt / np.sum(filt)

def matrix_inverse_sqrt(input, eps=1e-2):
    e, v = np.linalg.eigh(input)
    e = np.maximum(e, 0)
    e_rsqrt = 1.0 / np.sqrt(e + eps)
    return np.dot(v, np.dot(np.diag(e_rsqrt), v.T))

def spherical_kmeans_(x, n_clusters, max_iters=100, block_size=None, verbose=True, init=None):
    n_samples, n_features = x.shape
    if init is None:
        indices = np.random.choice(n_samples, n_clusters, replace=False)
        clusters = x[indices]
    else:
        clusters = init
    prev_sim = np.inf
    tmp = np.empty(n_samples)
    assign = np.empty(n_samples, dtype=np.int64)
    if block_size is None or block_size == 0:
        block_size = n_samples
    for n_iter in range(max_iters):
        for i in range(0, n_samples, block_size):
            end_i = min(i + block_size, n_samples)
            cos_sim = np.dot(x[i:end_i], clusters.T)
            tmp[i:end_i] = np.max(cos_sim, axis=1)
            assign[i:end_i] = np.argmax(cos_sim, axis=1)
        sim = np.mean(tmp)
        if (n_iter + 1) % 10 == 0 and verbose:
            print(f"  Spherical kmeans iter {n_iter+1}, objective {sim:.4f}")
        for j in range(n_clusters):
            index = assign == j
            if np.sum(index) == 0:
                idx = np.argmin(tmp)
                clusters[j] = x[idx]
                tmp[idx] = 1.
            else:
                c = np.mean(x[index], axis=0)
                clusters[j] = c / np.linalg.norm(c)
        if np.abs(prev_sim - sim) / (np.abs(sim) + 1e-20) < 1e-4:
            break
        prev_sim = sim
    return clusters, assign

def accuracy_score(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))


def zca_whitening(patches, eps=0.1):
    """
    ZCA whitening sur un batch de patches (N, D).
    Soustrait la moyenne, blanchit via la décomposition propre de la covariance.
    eps : regularisation pour éviter la division par zéro sur les petites valeurs propres.
    """
    patches = patches - patches.mean(axis=0, keepdims=True)
    cov = np.dot(patches.T, patches) / patches.shape[0]
    e, v = np.linalg.eigh(cov)
    e = np.maximum(e, 0)
    W_zca = v @ np.diag(1.0 / np.sqrt(e + eps)) @ v.T
    return patches @ W_zca.T, W_zca

def im2col(input, kernel_h, kernel_w, stride=1, padding=0):
    """Transforme l'input en matrice de patches pour conv matricielle."""
    N, C, H, W = input.shape
    if padding > 0:
        input = np.pad(input, ((0,0),(0,0),(padding,padding),(padding,padding)))
    H_out = (H + 2*padding - kernel_h) // stride + 1
    W_out = (W + 2*padding - kernel_w) // stride + 1
    shape   = (N, C, kernel_h, kernel_w, H_out, W_out)
    strides = (*input.strides[:2], *input.strides[2:], input.strides[2]*stride, input.strides[3]*stride)
    cols = np.lib.stride_tricks.as_strided(input, shape=shape, strides=strides)
    return cols.reshape(N, C * kernel_h * kernel_w, H_out * W_out)

def conv2d_fast(input, weight, bias=None, stride=1, padding=0, groups=1):
    """Conv2d vectorisée via im2col + matmul."""
    if not isinstance(weight, np.ndarray):
        weight = weight.values
    N, C, H, W = input.shape
    out_channels, in_channels_g, kH, kW = weight.shape

    if groups == 1:
        cols = im2col(input, kH, kW, stride=stride, padding=padding)
        W_flat = weight.reshape(out_channels, -1)
        out = np.tensordot(W_flat, cols, axes=([1],[1])).transpose(1,0,2)
        H_out = (H + 2*padding - kH) // stride + 1
        W_out = (W + 2*padding - kW) // stride + 1
        out = out.reshape(N, out_channels, H_out, W_out)
    else:
        H_out = (H + 2*padding - kH) // stride + 1
        W_out = (W + 2*padding - kW) // stride + 1
        out = np.zeros((N, out_channels, H_out, W_out))
        c_in_g  = C // groups
        c_out_g = out_channels // groups
        for g in range(groups):
            x_g = input[:, g*c_in_g:(g+1)*c_in_g]
            w_g = weight[g*c_out_g:(g+1)*c_out_g]
            cols = im2col(x_g, kH, kW, stride=stride, padding=padding)
            W_flat = w_g.reshape(c_out_g, -1)
            res = np.tensordot(W_flat, cols, axes=([1],[1])).transpose(1,0,2)
            out[:, g*c_out_g:(g+1)*c_out_g] = res.reshape(N, c_out_g, H_out, W_out)

    if bias is not None:
        out += bias
    return out


## 3. Kernels

In [3]:
def kernel_exp(x, alpha):
    return np.exp(alpha * (x - 1))

def kernel_poly(x, alpha=2):
    return np.power(x, alpha)

KERNELS = {"exp": kernel_exp, "poly": kernel_poly}

## 4. Loss

In [4]:
class CrossEntropyLoss:
    def __init__(self):
        self.item = None

    def __call__(self, output, target):
        self.output = output
        target = target.astype(int)
        self.target = target
        exp_scores = np.exp(output - np.max(output, axis=1, keepdims=True))
        probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        target_probs = probabilities[np.arange(len(output)), target]
        self.loss_value = -np.mean(np.log(target_probs + 1e-15))
        self.probabilities = probabilities
        self.item = self.loss_value
        return self

    def backward(self, x):
        gradient = self.probabilities.copy()
        gradient[np.arange(len(self.probabilities)), self.target] -= 1
        gradient /= len(self.probabilities)
        dW = np.dot(gradient.T, x)
        db = np.sum(gradient, axis=0, keepdims=True).T
        return dW, db

## 5. Paramètres & couches

In [5]:
class Parameters:
    def __init__(self, shape):
        self.values = np.random.randn(*shape)
        self.gradients = np.zeros(shape)
        self.requires_grad = True

    def zero_gradients(self):
        self.gradients.fill(0)

    @property
    def shape(self):
        return self.values.shape

    def __repr__(self):
        return f"Parameters(shape={self.values.shape})"


class CKNLayer:
    def __init__(self, in_channels, out_channels, filter_size,
                 subsampling, padding="SAME", dilation=1, groups=1,
                 subsampling_factor=1/math.sqrt(2),
                 kernel_func="exp", kernel_args=0.5):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.filter_size = (filter_size, filter_size)
        self.subsampling = subsampling
        self.padding = padding
        self.dilation = dilation
        self.groups = groups
        self.kernel_func = KERNELS[kernel_func]
        if isinstance(kernel_args, (int, float)):
            kernel_args = [kernel_args]
        if kernel_func == "exp":
            # kernel_args est sigma → on stocke alpha = 1/sigma²
            kernel_args = [1. / ka**2 for ka in kernel_args]
        self.kernel_args = kernel_args
        self.kappa = lambda x: KERNELS[kernel_func](x, *self.kernel_args)
        self.mode = 'train'
        self.patch_dim = in_channels * filter_size * filter_size

        weight_shape = (out_channels, in_channels, filter_size, filter_size)
        self.weight = Parameters(weight_shape)
        self.bias = None
        self.parameters_liste = [self.weight]

        # Gaussian pooling filter
        gauss = gaussian_filter(2 * subsampling + 1)
        self.pool_filter = np.outer(gauss, gauss).reshape(1, 1, 2*subsampling+1, 2*subsampling+1)
        self.pool_filter = np.broadcast_to(
            self.pool_filter, (out_channels, 1, 2*subsampling+1, 2*subsampling+1)).copy()

    def normalize(self):
        """Reprojette les filtres sur la sphère unité (nécessaire après SGD)."""
        W = self.weight.values.reshape(self.out_channels, -1)
        norms = np.linalg.norm(W, axis=1, keepdims=True)
        W = W / np.maximum(norms, EPS)
        self.weight.values[:] = W.reshape(self.weight.values.shape)

    def compute_lintrans(self):
        # FIX #1 : normaliser les filtres avant de calculer la gram matrix
        # Les filtres doivent être sur la sphère unité pour que kappa(M)[i,i] = kappa(1)
        # et que les valeurs propres soient dans un intervalle raisonnable.
        W = self.weight.values.reshape(self.out_channels, -1)
        W = W / np.maximum(np.linalg.norm(W, axis=1, keepdims=True), EPS)
        M = np.dot(W, W.T)          # M[i,j] = <w_i, w_j> ∈ [-1, 1]
        return matrix_inverse_sqrt(self.kappa(M))

    def _mult_layer(self, x_in, lintrans):
        """Applique la transformation linéaire (décorrélation) sur les canaux."""
        batch_size, out_c, H, W = x_in.shape
        # x_in : (N, out_c, H, W) → (N, out_c, H*W)
        x_flat = x_in.reshape(batch_size, out_c, -1)
        # lintrans : (out_c, out_c)  →  matmul sur chaque image du batch
        x_out = np.einsum('ij,bjk->bik', lintrans, x_flat)
        return x_out.reshape(batch_size, out_c, H, W)

    def conv_layer(self, x):
        """
        Implémente l'éq. (5) de l'article :
            ζ_k(z) = ||ψ_{k-1}(z)|| * kappa( <ψ̃_{k-1}(z), w_l> )
        où ψ̃ est la version ℓ2-normalisée du patch ψ.

        FIX #2 : on normalise les PATCHES avant le produit scalaire avec les filtres,
        et on multiplie ensuite par la norme du patch — ce n'est PAS équivalent à
        diviser la sortie de conv(x, W) par la norme du patch.
        """
        kH, kW = self.filter_size
        padding = kH // 2 if self.padding == "SAME" else 0
        N, C, H, W = x.shape

        # Extraire les patches : (N, C*kH*kW, H_out*W_out)
        cols = im2col(x, kH, kW, stride=1, padding=padding)

        # Norme de chaque patch : (N, 1, H_out*W_out)
        patch_norms = np.linalg.norm(cols, axis=1, keepdims=True)  # ||ψ(z)||

        # Patches normalisés : ψ̃(z)
        cols_norm = cols / np.maximum(patch_norms, EPS)

        # FIX #3 : normaliser aussi les filtres (ils doivent rester sur la sphère)
        W_flat = self.weight.values.reshape(self.out_channels, -1)
        W_flat = W_flat / np.maximum(np.linalg.norm(W_flat, axis=1, keepdims=True), EPS)

        # Produit scalaire <ψ̃(z), w_l> : (N, out_channels, H_out*W_out)
        H_out = (H + 2*padding - kH) // 1 + 1
        W_out = (W + 2*padding - kW) // 1 + 1
        dot = np.tensordot(W_flat, cols_norm, axes=([1],[1])).transpose(1,0,2)
        dot = dot.reshape(N, self.out_channels, H_out, W_out)

        # Appliquer kappa puis multiplier par la norme du patch
        patch_norms_2d = patch_norms.reshape(N, 1, H_out, W_out)
        x_out = patch_norms_2d * self.kappa(dot)
        return x_out

    def pool_layer(self, x):
        return conv2d_fast(x, self.pool_filter,
                        stride=self.subsampling, padding=self.subsampling,
                        groups=self.out_channels)

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.pool_layer(x)
        lintrans = self.compute_lintrans()
        return self._mult_layer(x, lintrans)

    def __call__(self, x):
        return self.forward(x)

    def extract_2d_patches(self, x):
        """Extrait les patches pour l'entraînement non-supervisé."""
        kH, kW = self.filter_size
        N, C, H, W = x.shape
        padding = kH // 2 if self.padding == "SAME" else 0
        cols = im2col(x, kH, kW, stride=1, padding=padding)
        # cols : (N, patch_dim, H_out*W_out) → (N*H_out*W_out, patch_dim)
        return cols.transpose(0, 2, 1).reshape(-1, self.patch_dim)

    def sample_patches(self, x, n_sampling_patches=1000):
        patches = self.extract_2d_patches(x)
        idx = np.random.choice(patches.shape[0],
                               min(patches.shape[0], n_sampling_patches),
                               replace=False)
        return patches[idx]

    def unsup_train(self, patches, use_zca=True):
        """
        Entraînement non-supervisé par sphérical k-means.
        Si use_zca=True, applique un ZCA whitening avant le k-means :
        cela force les filtres à capturer des directions de haute variance
        et non pas les basses fréquences dominantes (coins, gradients faibles).
        Sans ZCA sur CIFAR, le k-means converge systématiquement vers des filtres
        basses fréquences peu discriminants.
        """
        patches = patches.copy()
        # Centrer
        patches -= patches.mean(axis=0, keepdims=True)
        if use_zca and self.patch_dim > 2:
            patches, _ = zca_whitening(patches, eps=0.1)
        # Normaliser sur la sphère avant k-means
        patches = normalize_np(patches)
        block_size = None if self.patch_dim < 1000 else 10 * self.patch_dim
        weight = spherical_kmeans_(patches, self.out_channels, block_size=block_size)[0]
        self.weight.values[:] = weight.reshape(self.weight.values.shape)

    def parameters(self):
        return self.parameters_liste

    def __repr__(self):
        return (f"CKNLayer(in={self.in_channels}, out={self.out_channels}, "
                f"filter={self.filter_size}, subsampling={self.subsampling})")


class Linear:
    def __init__(self, in_features, out_features, alpha=0.0, fit_bias=True,
                 penalty="l2", maxiter=1000):
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.fit_bias = fit_bias
        self.penalty = penalty
        self.maxiter = maxiter
        self.parameters_liste = []
        self.weight = Parameters((out_features, in_features))
        self.parameters_liste.append(self.weight)
        if self.fit_bias:
            self.bias = Parameters((out_features,))
            self.bias.values = self.bias.values.reshape(-1, 1)
            self.parameters_liste.append(self.bias)
        else:
            self.bias = None

    def forward(self, input, scale_bias=1.0):
        out = np.dot(input, self.weight.values.T)
        if self.bias is not None:
            out += scale_bias * self.bias.values.ravel()
        return out

    def fit(self, x, y):
        criterion = CrossEntropyLoss()
        alpha = self.alpha * x.shape[1] / x.shape[0]
        scale_bias = np.sqrt(np.mean(np.mean(x ** 2, axis=-1))) if self.bias is not None else 1.0
        if self.bias is not None:
            alpha *= scale_bias ** 2
        self.real_alpha = alpha

        def eval_loss(w):
            w = w.reshape((self.out_features, -1))
            if self.bias is None:
                self.weight.values = w.copy()
                y_pred = np.dot(x, self.weight.values.T)
            else:
                self.weight.values = w[:, :-1].copy()
                self.bias.values = w[:, -1].copy()
                y_pred = np.dot(x, self.weight.values.T) + scale_bias * self.bias.values.squeeze()
            loss = criterion(y_pred, y)
            self.weight.gradients, self.bias.gradients = loss.backward(x)
            if alpha != 0.0:
                if self.penalty == "l2":
                    loss.item += 0.5 * alpha * np.linalg.norm(self.weight.values, ord=2) ** 2
                elif self.penalty == "l1":
                    loss.item += alpha * np.linalg.norm(self.weight.values, ord=1)
            return loss.item

        def eval_grad(w):
            dw = self.weight.gradients.copy()
            if alpha != 0.0 and self.penalty == "l2":
                dw += alpha * self.weight.values
            if self.bias is not None:
                dw = np.hstack((dw, self.bias.gradients.reshape(-1, 1)))
            return dw.ravel().astype("float64")

        w_init = self.weight.values.astype("float64")
        if self.bias is not None:
            bias_init = (1. / scale_bias) * self.bias.values.reshape(-1, 1).astype("float64")
            w_init = np.hstack((w_init, bias_init))

        result = scipy.optimize.minimize(
            eval_loss, w_init.ravel(), jac=eval_grad,
            method='L-BFGS-B',
            options={'maxiter': self.maxiter, 'disp': False}
        )
        w = result.x.reshape((self.out_features, -1))
        self.weight.zero_gradients()
        if self.bias is None:
            self.weight.values = w.copy()
        else:
            self.bias.zero_gradients()
            self.weight.values = w[:, :-1].copy()
            self.bias.values = (scale_bias * w[:, -1]).copy().reshape(-1, 1)

    def __call__(self, x):
        return self.forward(x)

    def parameters(self):
        return self.parameters_liste


## 6. Optimiseur SGD

In [6]:
class SGD:
    def __init__(self, param_groups, lr=0.01, momentum=0, weight_decay=0):
        # param_groups : liste de dicts {'params': [...], 'lr': ..., ...}
        self.param_groups = param_groups
        self.defaults = {'lr': lr, 'momentum': momentum, 'weight_decay': weight_decay}
        self.velocities = [
            [np.zeros_like(p.values) for p in g['params']]
            for g in self.param_groups
        ]

    def step(self):
        for group, vels in zip(self.param_groups, self.velocities):
            lr = group.get('lr', self.defaults['lr'])
            momentum = group.get('momentum', self.defaults['momentum'])
            weight_decay = group.get('weight_decay', self.defaults['weight_decay'])
            for param, vel in zip(group['params'], vels):
                if param.values.ndim == 1:
                    param.values = param.values.reshape(-1, 1)
                param.gradients = param.gradients.reshape(param.values.shape)
                param.gradients += weight_decay * param.values
                vel *= momentum
                vel -= lr * param.gradients
                param.values += vel

    def zero_grad(self):
        for group in self.param_groups:
            for p in group['params']:
                p.gradients.fill(0)

## 7. Modèle CKNet

In [7]:
class CKNSequential:
    def __init__(self, in_channels, out_channels_list, filter_sizes,
                 subsamplings, kernel_funcs=None, kernel_args_list=None, **kwargs):
        assert len(out_channels_list) == len(filter_sizes) == len(subsamplings)
        self.n_layers = len(out_channels_list)
        self.ckn_layers = []
        ch = in_channels
        for i in range(self.n_layers):
            kf = kernel_funcs[i] if kernel_funcs else "exp"
            ka = kernel_args_list[i] if kernel_args_list else 0.5
            layer = CKNLayer(ch, out_channels_list[i], filter_sizes[i],
                             subsamplings[i], kernel_func=kf, kernel_args=ka, **kwargs)
            self.ckn_layers.append(layer)
            ch = out_channels_list[i]

    def changemode(self, mode='inf'):
        for layer in self.ckn_layers:
            layer.mode = mode
        print(f'Mode -> {mode}')

    def forward(self, x):
        for layer in self.ckn_layers:
            x = layer(x)
        return x

    def representation(self, x, n=0):
        for i in range(n):
            x = self.ckn_layers[i](x)
        return x

    def normalize(self):
        for layer in self.ckn_layers:
            layer.normalize()

    def unsup_train_(self, data_loader, n_sampling_patches=100000):
        for i, layer in enumerate(self.ckn_layers):
            print(f'\n--- Training CKN layer {i+1}/{self.n_layers} ---')
            try:
                n_per_batch = (n_sampling_patches + len(data_loader) - 1) // len(data_loader)
            except:
                n_per_batch = 1000
            patches = np.zeros((n_sampling_patches, layer.patch_dim))
            n_patches = 0
            for data, _ in data_loader:
                data = self.representation(data, i)
                batch_patches = layer.sample_patches(data, n_per_batch)
                size = min(batch_patches.shape[0], n_sampling_patches - n_patches)
                patches[n_patches:n_patches + size] = batch_patches[:size]
                n_patches += size
                if n_patches >= n_sampling_patches:
                    break
            layer.unsup_train(patches[:n_patches])

    def __call__(self, x):
        return self.forward(x)


class CKNet:
    def __init__(self, nclass, in_channels, out_channels_list, kernel_sizes,
                 subsamplings, kernel_funcs=None, kernel_args_list=None,
                 image_size=32, fit_bias=True, alpha=0.0, maxiter=200, **kwargs):
        self.features = CKNSequential(
            in_channels, out_channels_list, kernel_sizes, subsamplings,
            kernel_funcs, kernel_args_list, **kwargs)
        out_ch = out_channels_list[-1]

        # FIX #4 : calcul correct de la taille de la feature map finale.
        # Chaque couche fait un pooling avec stride = subsampling, donc la taille
        # est divisée par subsampling à chaque couche (arrondi supérieur avec ceil).
        spatial = image_size
        for s in subsamplings:
            spatial = math.ceil(spatial / s)
        self.out_features = spatial * spatial * out_ch

        self.nclass = nclass
        self.classifier = Linear(self.out_features, nclass, fit_bias=fit_bias,
                                 alpha=alpha, maxiter=maxiter)

    def representation(self, x):
        return self.features(x).reshape(x.shape[0], -1)

    def forward(self, x):
        return self.classifier(self.representation(x))

    def __call__(self, x):
        return self.forward(x)

    def normalize(self):
        self.features.normalize()

    def unsup_train_ckn(self, data_loader, n_sampling_patches=150000):
        self.features.unsup_train_(data_loader, n_sampling_patches)

    def unsup_train_classifier(self, data_loader):
        X_enc, Y_enc = self._encode(data_loader)
        self.classifier.fit(X_enc, Y_enc)

    def _encode(self, data_loader):
        X_list, Y_list = [], []
        for data, target in data_loader:
            X_list.append(self.representation(data))
            Y_list.append(target)
        return np.concatenate(X_list), np.concatenate(Y_list)

    def get_parameters(self):
        params = []
        for layer in self.features.ckn_layers:
            params.extend(layer.parameters())
        params.extend(self.classifier.parameters())
        return params


# Architectures disponibles
class CKN2(CKNet):
    """2 couches : [32, 32], filtres [3, 3], subsampling [2, 6]"""
    def __init__(self, alpha=0.0, maxiter=5000, **kw):
        super().__init__(10, 3, [32, 32], [3, 3], [2, 6],
                         kernel_funcs=['exp','exp'],
                         kernel_args_list=[1/np.sqrt(3), 1/np.sqrt(3)],
                         fit_bias=True, alpha=alpha, maxiter=maxiter, **kw)

class CKN3(CKNet):
    """3 couches : [64, 64, 128], filtres [3, 3, 3], subsampling [2, 2, 2]"""
    def __init__(self, alpha=0.0, maxiter=5000, **kw):
        super().__init__(10, 3, [64, 64, 128], [3, 3, 3], [2, 2, 2],
                         kernel_funcs=['exp','exp','exp'],
                         kernel_args_list=[0.6, 0.6, 0.6],
                         fit_bias=True, alpha=alpha, maxiter=maxiter, **kw)

class CKN5(CKNet):
    """5 couches : [64,32,64,32,64], filtres [3,1,3,1,3], subsampling [2,1,2,1,3]"""
    def __init__(self, alpha=0.0, maxiter=5000, **kw):
        super().__init__(10, 3, [64,32,64,32,64], [3,1,3,1,3], [2,1,2,1,3],
                         kernel_funcs=['exp','poly','exp','poly','exp'],
                         kernel_args_list=[0.5, 2, 0.5, 2, 0.5],
                         fit_bias=True, alpha=alpha, maxiter=maxiter, **kw)

MODELS = {'ckn2': CKN2, 'ckn3': CKN3, 'ckn5': CKN5}


## 8. Chargement des données

In [8]:
DATA_PATH = './data/'

Xtr_raw = np.array(pd.read_csv(f'{DATA_PATH}Xtr.csv', header=None, sep=',',
                                usecols=range(3072), encoding='unicode_escape'))
Ytr_raw = np.array(pd.read_csv(f'{DATA_PATH}Ytr.csv', sep=',', usecols=[1])).squeeze()
Xtr_raw = Xtr_raw.reshape(-1, 3, 32, 32).astype(np.float32)

# Les données sont déjà pré-normalisées par le challenge (valeurs dans [-0.46, 0.40]).
# On applique quand même une normalisation par canal sur le train set
# pour que la moyenne soit exactement 0 et la std exactement 1,
# ce qui est attendu par le CKN (patches normalisés sur la sphère).
channel_mean = Xtr_raw.mean(axis=(0, 2, 3), keepdims=True)  # (1, 3, 1, 1)
channel_std  = Xtr_raw.std(axis=(0, 2, 3), keepdims=True)   # (1, 3, 1, 1)
Xtr_raw = (Xtr_raw - channel_mean) / (channel_std + 1e-6)

# Split train / dev
val_ratio = 0.2
n = len(Xtr_raw)
n_val = int(n * val_ratio)
idx = np.random.permutation(n)

X_dev,   Y_dev   = Xtr_raw[idx[:n_val]],  Ytr_raw[idx[:n_val]]
X_train, Y_train = Xtr_raw[idx[n_val:]],  Ytr_raw[idx[n_val:]]

print(f"X_train: {X_train.shape}  Y_train: {Y_train.shape}")
print(f"X_dev:   {X_dev.shape}    Y_dev:   {Y_dev.shape}")
print(f"Après normalisation — min: {X_train.min():.4f}, max: {X_train.max():.4f}, "
      f"mean: {X_train.mean():.4f}, std: {X_train.std():.4f}")


X_train: (4000, 3, 32, 32)  Y_train: (4000,)
X_dev:   (1000, 3, 32, 32)    Y_dev:   (1000,)
Après normalisation — min: -9.5484, max: 10.6810, mean: -0.0002, std: 1.0006


In [9]:
print(X_train.min(), X_train.max())

-9.548441 10.680965


## 9. DataLoader

In [10]:
class DataLoader:
    """DataLoader minimaliste compatible avec le training loop."""
    def __init__(self, X, y=None, batch_size=64, shuffle=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_samples = len(X)

    def __len__(self):
        return math.ceil(self.num_samples / self.batch_size)

    def __iter__(self):
        idx = np.random.permutation(self.num_samples) if self.shuffle else np.arange(self.num_samples)
        for start in range(0, self.num_samples, self.batch_size):
            batch_idx = idx[start:start + self.batch_size]
            if self.y is not None:
                yield self.X[batch_idx], self.y[batch_idx]
            else:
                yield self.X[batch_idx]

## 10. Configuration de l'expérience

In [11]:
# ── Hyperparamètres ──────────────────────────────────────────────
MODEL_NAME         = 'ckn3'   # 'ckn2' | 'ckn3' | 'ckn5'
BATCH_SIZE         = 256
ALPHA              = 0.001    # régularisation L2 du classifieur
N_SAMPLING_PATCHES = 300000   # augmenté : plus de patches → meilleur k-means
MAXITER_BFGS       = 2000     # augmenté : BFGS bien convergé est crucial
# ─────────────────────────────────────────────────────────────────

train_loader = DataLoader(X_train, Y_train, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(X_dev,   Y_dev,   batch_size=BATCH_SIZE, shuffle=False)

model = MODELS[MODEL_NAME](alpha=ALPHA, maxiter=MAXITER_BFGS)
n_params = sum(np.prod(p.values.shape) for p in model.get_parameters())
print(f"Modèle : {MODEL_NAME}  |  Paramètres : {n_params:,}")
print(f"out_features : {model.out_features}")


Modèle : ckn3  |  Paramètres : 132,810
out_features : 2048


## 11. Entraînement

In [12]:
# ── Phase 1 : entraînement non-supervisé des filtres CKN ──────────
print("=" * 50)
print("Phase 1 — Unsupervised training des couches CKN (avec ZCA whitening)")
print("=" * 50)
model.unsup_train_ckn(train_loader, N_SAMPLING_PATCHES)

weights = [layer.weight.values.copy() for layer in model.features.ckn_layers]
with open('weights.pkl', 'wb') as f:
    pickle.dump(weights, f)
print("\nPoids sauvegardés dans weights.pkl")


Phase 1 — Unsupervised training des couches CKN (avec ZCA whitening)

--- Training CKN layer 1/3 ---
  Spherical kmeans iter 10, objective 0.5577
  Spherical kmeans iter 20, objective 0.5617
  Spherical kmeans iter 30, objective 0.5632

--- Training CKN layer 2/3 ---
  Spherical kmeans iter 10, objective 0.5279
  Spherical kmeans iter 20, objective 0.5322
  Spherical kmeans iter 30, objective 0.5332

--- Training CKN layer 3/3 ---
  Spherical kmeans iter 10, objective 0.6638
  Spherical kmeans iter 20, objective 0.6667

Poids sauvegardés dans weights.pkl


In [12]:
# ── Phase 2 : classifieur linéaire par L-BFGS ─────────────────────
# L'article entraîne uniquement le classifieur (SVM / régression logistique L-BFGS)
# sur les features CKN figées. Les couches CKN ne sont PAS mises à jour ici.
# C'est plus simple, plus rapide, et plus fidèle à l'article.

with open('weights.pkl', 'rb') as f:
    loaded_weights = pickle.load(f)

for layer, w in zip(model.features.ckn_layers, loaded_weights):
    layer.weight.values = w.copy()

print("=" * 50)
print("Phase 2 — Classifieur L-BFGS sur features CKN figées")
print("=" * 50)

from timeit import default_timer as timer
tic = timer()
model.unsup_train_classifier(train_loader)
elapsed = timer() - tic
print(f"BFGS terminé en {elapsed:.1f}s")

# Évaluation
def evaluate(loader, split_name):
    preds, targets = [], []
    for X_batch, y_batch in loader:
        preds.append(np.argmax(model(X_batch), axis=1))
        targets.append(y_batch)
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    acc = np.mean(preds == targets)
    print(f"  {split_name:10s} accuracy : {acc*100:.2f}%")
    return acc

train_acc = evaluate(train_loader, "Train")
dev_acc   = evaluate(dev_loader,   "Dev")


Phase 2 — Classifieur L-BFGS sur features CKN figées


C:\Users\paulr\AppData\Local\Temp\ipykernel_22892\3149201998.py:237: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = scipy.optimize.minimize(


BFGS terminé en 99.9s
  Train      accuracy : 76.12%
  Dev        accuracy : 62.00%


tests:

In [13]:
# ── Vérification santé du modèle après entraînement non-supervisé ──────────────
X_test = X_train[:16]

print("=== Propagation forward couche par couche ===")
x = X_test.copy()
print(f"Input: shape={x.shape}, mean={x.mean():.4f}, std={x.std():.4f}")

for i, layer in enumerate(model.features.ckn_layers):
    x = layer(x)
    var_inter = x.reshape(16, -1).var(axis=0).mean()
    print(f"Layer {i+1}: shape={x.shape}, mean={x.mean():.4f}, "
          f"std={x.std():.4f}, var_inter_images={var_inter:.6f}")

print("\n=== Vérification filtres (norme ≈ 1) ===")
for i, layer in enumerate(model.features.ckn_layers):
    W = layer.weight.values.reshape(layer.out_channels, -1)
    norms = np.linalg.norm(W, axis=1)
    print(f"Layer {i+1}: norme filtres — min={norms.min():.4f}, "
          f"max={norms.max():.4f}, mean={norms.mean():.4f}")

print("\n=== Vérification lintrans (pas de NaN/Inf) ===")
for i, layer in enumerate(model.features.ckn_layers):
    lt = layer.compute_lintrans()
    print(f"Layer {i+1}: lintrans shape={lt.shape}, "
          f"has_nan={np.any(np.isnan(lt))}, has_inf={np.any(np.isinf(lt))}, "
          f"norm={np.linalg.norm(lt):.4f}")


=== Propagation forward couche par couche ===
Input: shape=(16, 3, 32, 32), mean=0.0017, std=1.0814
Layer 1: shape=(16, 64, 16, 16), mean=0.1508, std=0.1554, var_inter_images=0.019164
Layer 2: shape=(16, 64, 8, 8), mean=0.1297, std=0.1525, var_inter_images=0.008237
Layer 3: shape=(16, 128, 4, 4), mean=0.0950, std=0.1406, var_inter_images=0.003204

=== Vérification filtres (norme ≈ 1) ===
Layer 1: norme filtres — min=1.0000, max=1.0000, mean=1.0000
Layer 2: norme filtres — min=1.0000, max=1.0000, mean=1.0000
Layer 3: norme filtres — min=1.0000, max=1.0000, mean=1.0000

=== Vérification lintrans (pas de NaN/Inf) ===
Layer 1: lintrans shape=(64, 64), has_nan=False, has_inf=False, norm=8.4358
Layer 2: lintrans shape=(64, 64), has_nan=False, has_inf=False, norm=8.6362
Layer 3: lintrans shape=(128, 128), has_nan=False, has_inf=False, norm=14.5983


## 12. Résultats

In [14]:
# ── Résumé des résultats ─────────────────────────────────────────
print("=" * 50)
print(f"Résultats finaux")
print("=" * 50)
print(f"  Modèle      : {MODEL_NAME}")
print(f"  Alpha       : {ALPHA}")
print(f"  BFGS maxiter: {MAXITER_BFGS}")
print(f"  Patches     : {N_SAMPLING_PATCHES:,}")
print(f"  Train acc   : {train_acc*100:.2f}%")
print(f"  Dev acc     : {dev_acc*100:.2f}%")
print(f"  Gap         : {(train_acc - dev_acc)*100:.2f}%")


Résultats finaux
  Modèle      : ckn3
  Alpha       : 0.001
  BFGS maxiter: 2000
  Patches     : 300,000
  Train acc   : 76.12%
  Dev acc     : 62.00%
  Gap         : 14.12%
